In [1]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [2]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [3]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [4]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [5]:
from rag_helper import RAGBase
from ingest import load_faq_data, build_index


documents = load_faq_data()
index = build_index(documents) 

In [6]:
import json

def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [7]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

messages.extend(response.output)
has_function_calls = False

In [8]:
for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

function_call: search {"query":"join course discovered course can I join late enrollment FAQ"}
function_call: search {"query":"course discovered can I enroll after start FAQ late join"}
function_call: search {"query":"enrollment after course start join course FAQ"}


In [10]:
messages

[{'role': 'developer',
  'content': "You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore."},
 {'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"join course discovered course can I join late enrollment FAQ"}', call_id='call_wKT69PtDYv0JXar68iqs1FGd', name='search', type='function_call', id='fc_09fb69f79b5dcb16006a2ea126d2b0819aabc37ba686ed800b', namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"course discovered can I enroll after start FAQ late join"}', call_id='call_HAXULvni

In [12]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

it = 1

while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool],
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT:")
            print(item.content[0].text)

    it = it + 1
    if has_function_calls == False:
        break

iteration #1...
function_call: search {"query":"join course discovered course can I join enrollment late registration FAQ"}
iteration #2...
ASSISTANT:
Yes — you can still join the course.

If you want to get a certificate, the key thing is to submit your project while submissions are still being accepted. If you’re just learning along the way, you can start even if you discovered it late.

If you want, I can also explain how certificates and project submissions work.


In [13]:
def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

In [14]:
agent_loop(instructions, "How do I run Olama locally?")

iteration #1...
function_call: search {"query":"Ollama run locally install local model server FAQ"}
function_call: search {"query":"Olama local run FAQ"}
function_call: search {"query":"Ollama local usage command run model FAQ"}
iteration #2...
ASSISTANT:
To run **Ollama locally**:

1. **Install Ollama**
   - macOS: download and install from https://ollama.com/download
   - Windows: download the `.msi` from https://ollama.com/download
   - Linux:
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

2. **Start a local model**
   ```bash
   ollama run llama3
   ```
   This will download the model if needed, start it locally, and open a chat-like prompt.

3. **Check that the local server is running**
   ```bash
   curl http://localhost:11434
   ```
   You should get a response showing the server is up.

4. **Use it from Python**
   ```bash
   pip install ollama
   ```

   Example:
   ```python
   import ollama

   response = ollama.chat(
       model='llama3',
       

'To run **Ollama locally**:\n\n1. **Install Ollama**\n   - macOS: download and install from https://ollama.com/download\n   - Windows: download the `.msi` from https://ollama.com/download\n   - Linux:\n     ```bash\n     curl -fsSL https://ollama.com/install.sh | sh\n     ```\n\n2. **Start a local model**\n   ```bash\n   ollama run llama3\n   ```\n   This will download the model if needed, start it locally, and open a chat-like prompt.\n\n3. **Check that the local server is running**\n   ```bash\n   curl http://localhost:11434\n   ```\n   You should get a response showing the server is up.\n\n4. **Use it from Python**\n   ```bash\n   pip install ollama\n   ```\n\n   Example:\n   ```python\n   import ollama\n\n   response = ollama.chat(\n       model=\'llama3\',\n       messages=[{"role": "user", "content": "Hello!"}]\n   )\n\n   print(response[\'message\'][\'content\'])\n   ```\n\nIf you’re getting a connection issue in a notebook or after restarting, you may need to restart the Ollama

In [15]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "I just discovered the course. Can I join it?")

iteration #1...
function_call: search {"query":"join the course discovered late enrollment can I join course FAQ"}
iteration #2...
ASSISTANT:
Yes — you can still join the course even if you just discovered it.

One important caveat: if you want a certificate, you need to submit your project while the course is still accepting submissions. If you’re only interested in learning, you can still follow along.

If you’d like, I can also help with:
- whether you can join self-paced,
- how certificates work,
- or what to do after you start.


'Yes — you can still join the course even if you just discovered it.\n\nOne important caveat: if you want a certificate, you need to submit your project while the course is still accepting submissions. If you’re only interested in learning, you can still follow along.\n\nIf you’d like, I can also help with:\n- whether you can join self-paced,\n- how certificates work,\n- or what to do after you start.'

In [16]:
agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit chess opening queen's gambit"}
iteration #2...
function_call: search {"query":"queen's gambit chess opening definition"}
iteration #3...
ASSISTANT:
The **Queen’s Gambit** is a chess opening that starts with:

1. **d4 d5**
2. **c4**

White offers the c-pawn to try to gain control of the center and encourage Black’s d-pawn to move or be exchanged. It’s one of the oldest and most respected chess openings.

There are two main ideas:
- **Queen’s Gambit Accepted**: Black takes the pawn with `dxc4`
- **Queen’s Gambit Declined**: Black does not take and instead supports the center

If you meant **“queen’s gambit”** as in the chess opening, that’s it. If you want, I can also explain:
- the main lines,
- why it’s called a gambit,
- or how to play it as White or Black.


'The **Queen’s Gambit** is a chess opening that starts with:\n\n1. **d4 d5**\n2. **c4**\n\nWhite offers the c-pawn to try to gain control of the center and encourage Black’s d-pawn to move or be exchanged. It’s one of the oldest and most respected chess openings.\n\nThere are two main ideas:\n- **Queen’s Gambit Accepted**: Black takes the pawn with `dxc4`\n- **Queen’s Gambit Declined**: Black does not take and instead supports the center\n\nIf you meant **“queen’s gambit”** as in the chess opening, that’s it. If you want, I can also explain:\n- the main lines,\n- why it’s called a gambit,\n- or how to play it as White or Black.'

In [17]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit"}
iteration #2...
function_call: search {"query":"queen's gambit opening chess course FAQ"}
iteration #3...
ASSISTANT:
I couldn’t find a course FAQ entry for “queen gambit,” so it looks like this may be off-topic or not covered by the course database.

If you meant the chess opening, the FAQ doesn’t have information about it. If you meant something else related to the course, please clarify and I’ll check again. Are there other areas you want to explore?


'I couldn’t find a course FAQ entry for “queen gambit,” so it looks like this may be off-topic or not covered by the course database.\n\nIf you meant the chess opening, the FAQ doesn’t have information about it. If you meant something else related to the course, please clarify and I’ll check again. Are there other areas you want to explore?'